In [25]:
import os
os.listdir()

['.ipynb_checkpoints', 'CLEANING', 'EDA']

In [2]:
import pandas as pd
import re
import nltk

In [3]:
nltk.download('stopwords')  
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\vijay\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\vijay\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [4]:
from nltk.corpus import stopwords

In [5]:
df = pd.read_csv('review.csv')

In [6]:
df.head()

,id,asins,brand,categories,colors,dateAdded,dateUpdated,dimension,ean,keys,...,reviews.rating,reviews.sourceURLs,reviews.text,reviews.title,reviews.userCity,reviews.userProvince,reviews.username,sizes,upc,weight
0,AVpe7AsMilAPnD_xQ78G,B00QJDU3KY,Amazon,"Amazon Devices,mazon.co.uk",NaN,2016-03-08T20:21:53Z,2017-07-18T23:52:58Z,169 mm x 117 mm x 9.1 mm,NaN,kindlepaperwhite/b00qjdu3ky,...,5.0,https://www.amazon.com/Kindle-Paperwhite-High-...,I initially had trouble deciding between the p...,"Paperwhite voyage, no regrets!",NaN,NaN,Cristina M,NaN,NaN,205 grams
1,AVpe7AsMilAPnD_xQ78G,B00QJDU3KY,Amazon,"Amazon Devices,mazon.co.uk",NaN,2016-03-08T20:21:53Z,2017-07-18T23:52:58Z,169 mm x 117 mm x 9.1 mm,NaN,kindlepaperwhite/b00qjdu3ky,...,5.0,https://www.amazon.com/Kindle-Paperwhite-High-...,Allow me to preface this with a little history...,One Simply Could Not Ask For More,NaN,NaN,Ricky,NaN,NaN,205 grams
2,AVpe7AsMilAPnD_xQ78G,B00QJDU3KY,Amazon,"Amazon Devices,mazon.co.uk",NaN,2016-03-08T20:21:53Z,2017-07-18T23:52:58Z,169 mm x 117 mm x 9.1 mm,NaN,kindlepaperwhite/b00qjdu3ky,...,4.0,https://www.amazon.com/Kindle-Paperwhite-High-...,I am enjoying it so far. Great for reading. Ha...,Great for those that just want an e-reader,NaN,NaN,Tedd Gardiner,NaN,NaN,205 grams
3,AVpe7AsMilAPnD_xQ78G,B00QJDU3KY,Amazon,"Amazon Devices,mazon.co.uk",NaN,2016-03-08T20:21:53Z,2017-07-18T23:52:58Z,169 mm x 117 mm x 9.1 mm,NaN,kindlepaperwhite/b00qjdu3ky,...,5.0,https://www.amazon.com/Kindle-Paperwhite-High-...,I bought one of the first Paperwhites and have...,Love / Hate relationship,NaN,NaN,Dougal,NaN,NaN,205 grams
4,AVpe7AsMilAPnD_xQ78G,B00QJDU3KY,Amazon,"Amazon Devices,mazon.co.uk",NaN,2016-03-08T20:21:53Z,2017-07-18T23:52:58Z,169 mm x 117 mm x 9.1 mm,NaN,kindlepaperwhite/b00qjdu3ky,...,5.0,https://www.amazon.com/Kindle-Paperwhite-High-...,I have to say upfront - I don't like coroporat...,I LOVE IT,NaN,NaN,Miljan David Tanic,NaN,NaN,205 grams


In [7]:
print("=" * 50)
print(f"Loaded {df.shape[0]} rows and {df.shape[1]} columns")
print("=" * 50)

Loaded 1597 rows and 27 columns


In [8]:
keep_cols = [
    'name',               
    'reviews.date',       
    'reviews.rating',     
    'reviews.title',      
    'reviews.text',       
    'reviews.username',   
    'reviews.doRecommend' 
]
df = df[keep_cols].copy()

In [9]:
df.columns = ['product', 'date', 'rating', 'title', 'review_text', 'username', 'recommend']

In [10]:
print(f"{len(df.columns)} useful columns")
print("Columns:", df.columns.tolist())

7 useful columns
Columns: ['product', 'date', 'rating', 'title', 'review_text', 'username', 'recommend']


In [11]:
before = len(df)
df = df.dropna(subset=['review_text'])   
df = df.dropna(subset=['rating'])      
df['rating'] = df['rating'].astype(int)

In [12]:
print(f"{before - len(df)} rows with missing data")
print(f"Rows remaining: {len(df)}")

420 rows with missing data
Rows remaining: 1177


In [13]:
df['title']     = df['title'].fillna('No title')
df['username']  = df['username'].fillna('Anonymous')
df['recommend'] = df['recommend'].fillna(0)

In [14]:
print(f"Filled missing values in title, username, recommend")

Filled missing values in title, username, recommend


In [15]:
def label_sentiment(rating):
    if rating >= 4:
        return 'positive'
    elif rating == 3:
        return 'neutral'
    else:
        return 'negative'

df['sentiment'] = df['rating'].apply(label_sentiment)

In [16]:
print(f"Created sentiment labels from star ratings")
print("Sentiment distribution:")
print(df['sentiment'].value_counts().to_string())

Created sentiment labels from star ratings
Sentiment distribution:
sentiment
positive    977
neutral     124
negative     76


In [17]:
stopwords.words('english')

['a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 'her',
 'here',
 'hers',
 'herself',
 "he's",
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 'if',
 "i'll",
 "i'm",
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 "i've",
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [18]:
stop_words = set(stopwords.words('english'))

In [19]:
def clean_text(text):
    text = str(text)

    text = text.lower()   # 1. lowercase
  
    text = re.sub(r'http\S+|www\S+', '', text)    # 2. Remove (URLs)

    text = re.sub(r'<.*?>', '', text)    # 3. Remove HTML tags (some reviews have these)

    text = re.sub(r'[^a-z\s]', '', text)    # 4. Remove everything except normal English letters

    text = re.sub(r'\s+', ' ', text).strip()    # 5. Remove extra whitespace 

    tokens = text.split()    # 6. Split into individual words, then remove stopwords
    
    tokens = [word for word in tokens if word not in stop_words]

    return ' '.join(tokens)    # 7. Join the words back into a sentence


In [20]:
df['clean_text'] = df['review_text'].apply(clean_text)

print(f"Cleaned all review texts!")

# Show before/after example
print("\nExample of cleaning:")
print("   BEFORE:", df['review_text'].iloc[0][:120])
print("   AFTER: ", df['clean_text'].iloc[0][:120])


Cleaned all review texts!

Example of cleaning:
   BEFORE: I initially had trouble deciding between the paperwhite and the voyage because reviews more or less said the same thing:
   AFTER:  initially trouble deciding paperwhite voyage reviews less said thing paperwhite great spending money go voyagefortunatel


In [21]:
before = len(df)
df = df[df['clean_text'].str.len() > 10]

print(f"{before - len(df)} too-short reviews")

15 too-short reviews


In [22]:
print("\n" + "-" * 20)
print("Completed cleaning")
print("-" * 20)


--------------------
Completed cleaning
--------------------


In [23]:
print(f"Final dataset: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nColumns in your clean dataset:")
for col in df.columns:
    print(f"  • {col}")
print(f"\nSentiment counts:")
print(df['sentiment'].value_counts().to_string())

Final dataset: 1162 rows × 9 columns

Columns in your clean dataset:
  • product
  • date
  • rating
  • title
  • review_text
  • username
  • recommend
  • sentiment
  • clean_text

Sentiment counts:
sentiment
positive    964
neutral     124
negative     74


In [24]:
df.to_csv('cleaned_reviews.csv', index=False)
print(f"\nSaved as: cleaned_reviews.csv")
print("Load this file in for EDA")



Saved as: cleaned_reviews.csv
Load this file in for EDA
